# Verify installed host revision: exact preservation and explicit encoding lineage


66.04 migration verification. Require every withdrawal cell, all original gene evidence, exact unambiguous and unrelated installed values, exact retained cache text and unchanged parasite/mouse inference inputs. Keep the 13 source/legacy decimal representations separate. Earlier pandas float assertions used their default tolerance; this notebook explicitly measures raw binary differences and uses check_exact=True for preservation claims.

In [1]:
import pandas as pd, numpy as np, json, hashlib
from pathlib import Path
from starplast import host as H, organisms as O
base=Path('results/host_symbol_mapping_2026_10_08')
migration=base/'migration'
manifest=json.loads((migration/'manifest.json').read_text())
backup=Path('/media/carruthers/mnt3/claude/toxoplasma_projects/datasets/installed_revision_backups/66_04_2026_10_08')
data=Path('starplast/data')
before=pd.read_parquet(backup/O.HOST_TABLES[O.HUMAN])
after=pd.read_parquet(data/O.HOST_TABLES[O.HUMAN])
ledger=H.projection_withdrawals()
assert len(ledger)==117 and ledger.host_id.nunique()==39
assert ledger.field.value_counts().to_dict()=={f:39 for f in ('rhoptry_discharge_score','rhoptry_discharge_beta','rhoptry_discharge_fdr')}
expected=before.copy(deep=True)
for row in ledger.to_dict('records'):
    position=expected.index[expected.host_id.eq(row['host_id'])]
    assert len(position)==1 and expected.at[position[0],row['field']]==row['old_value']
    expected.at[position[0],row['field']]=np.nan
pd.testing.assert_frame_equal(after,expected,check_exact=True)
assert len(after)==20989
frozen=pd.read_parquet(base/'rhoptry_gene_evidence.parquet')
preserved=pd.read_parquet(data/'deposit_host_k562_rhoptry_gene_evidence.parquet')
pd.testing.assert_frame_equal(frozen,preserved,check_exact=True)
assert len(preserved)==20010
old_lines=(backup/'deposit_host_k562_rhoptry_screen.tsv').read_text().splitlines(keepends=True)
id_pos=old_lines[0].rstrip('\n').split('\t').index('host_id')
expected_lines=[old_lines[0]]+[line for line in old_lines[1:] if line.rstrip('\n').split('\t')[id_pos] not in set(ledger.host_id)]
assert (data/'deposit_host_k562_rhoptry_screen.tsv').read_text()==''.join(expected_lines)
for path,digest in manifest['original_runtime_sha256'].items():
    file=Path(path)
    if file.name not in ('hs_host_proteins.parquet','deposit_host_k562_rhoptry_screen.tsv'):
        assert hashlib.sha256(file.read_bytes()).hexdigest()==digest
print('117 cells withdrawn, all other installed cells exact, all 20,010 original gene records exact, retained TSV lines exact, mouse/parasite graph/feature/track-record/claim inputs unchanged.')

117 cells withdrawn, all other installed cells exact, all 20,010 original gene records exact, retained TSV lines exact, mouse/parasite graph/feature/track-record/claim inputs unchanged.


In [2]:
candidate=pd.read_parquet(base/'rhoptry_corrected_projection_candidate.parquet').set_index('host_id')
installed=before.set_index('host_id')
precision={}
for field in ('rhoptry_discharge_score','rhoptry_discharge_beta','rhoptry_discharge_fdr'):
    raw=candidate[field].to_numpy(dtype=float)
    legacy=installed.loc[candidate.index,field].to_numpy(dtype=float)
    different=raw!=legacy
    # Explicit encoding diagnostic; do not overwrite either original representation.
    normalized=np.array([float(format(float(v),'.15g')) for v in raw])
    equivalent=(raw==legacy)|(normalized==legacy)
    precision[field]={'rows':len(raw),'source_vs_legacy_binary_differences':int(different.sum()),'maximum_absolute_difference':float(np.max(np.abs(raw-legacy))),'not_exact_or_declared_decimal_15':int((~equivalent).sum())}
    print(field,'pre-existing source/legacy encoding gaps',int((~equivalent).sum()))
    pd.DataFrame({'host_id':candidate.index,'source_value':raw,'legacy_installed_value':legacy,'decimal_15_value':normalized,'binary_equal':~different,'exact_or_decimal_15':equivalent}).pipe(lambda frame: pd.testing.assert_frame_equal(frame,pd.read_parquet(migration/(field+'_source_legacy_encoding_rows.parquet')),check_exact=True))
precision['withdrawn_decimal_encoding_cells']=int(ledger.legacy_value_encoding.ne('exact_source_float').sum())
assert json.loads((migration/'precision_review.json').read_text())==precision
print(json.dumps(precision,indent=2))

rhoptry_discharge_score pre-existing source/legacy encoding gaps 236
rhoptry_discharge_beta pre-existing source/legacy encoding gaps 0
rhoptry_discharge_fdr pre-existing source/legacy encoding gaps 0
{
  "rhoptry_discharge_score": {
    "rows": 18700,
    "source_vs_legacy_binary_differences": 7221,
    "maximum_absolute_difference": 2.220446049250313e-16,
    "not_exact_or_declared_decimal_15": 236
  },
  "rhoptry_discharge_beta": {
    "rows": 18700,
    "source_vs_legacy_binary_differences": 1,
    "maximum_absolute_difference": 6.938893903907228e-18,
    "not_exact_or_declared_decimal_15": 0
  },
  "rhoptry_discharge_fdr": {
    "rows": 18700,
    "source_vs_legacy_binary_differences": 4,
    "maximum_absolute_difference": 1.3877787807814457e-17,
    "not_exact_or_declared_decimal_15": 0
  },
  "withdrawn_decimal_encoding_cells": 13
}


The first builder audit's phrase exact installed scores described agreement under pandas default float tolerances. This precision review supplies strict binary counts and declared decimal-equivalence checks; do not reuse that phrase for source-versus-legacy binary identity. Preservation across this migration is strictly exact: no unambiguous installed value changed and raw source values remain exact. Some unambiguous legacy source/cache encodings differ beyond this specific decimal representation; every exact value/difference is retained as a pre-existing transform-review gap, not silently normalized. No unambiguous installed value was changed. Historical coverage uses only the verified 117-cell ledger, and a changed original value is rejected by a regression test. This is a source/entity correction, not an accuracy/calibration claim or a promotion of GTEx/Atlas candidates.